In [ ]:
#from transformers import AutoProcessor, Llama4ForConditionalGeneration
import torch
from PIL import Image
import glob
import os
import sys
sys.path.append('/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/LlavaGuard')
import torch
from transformers import Llama4ForConditionalGeneration, AutoProcessor
from llavaguard.taxonomy.PEGI.PEGI_Graph import policy_graph
#from pegiguard.data.prompts import build_prompt, build_prompt_violence, build_prompt_criminal, build_prompt_disturbing, build_prompt_substances, build_prompt_hate, build_prompt_nudity, build_prompt_selfharm

In [ ]:
model_id = "meta-llama/Llama-4-Scout-17B-16E-Instruct"
processor = AutoProcessor.from_pretrained(model_id)

model = Llama4ForConditionalGeneration.from_pretrained(
    model_id,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

In [ ]:
import json

system_rule = (
    "You are a careful, literal safety reviewer. "
    "Output ONLY a single JSON object exactly matching the schema provided. "
    "No extra text, no markdown, no explanations outside JSON."
)
#trying out different prompts styles

user_prompt = """Top-level Category: S1: Violence
Subcategory to assess: "9. Gross Violence"

Definition:
- Edge: e.g.
- PEGI rating: PEGI 18
- Examples:  "Detailed scenes of excessive gore", "Motiveless or gratuitous violence", "Violence against defenseless individuals", "Torture of humans"

Assessment steps:
  1) Based on the examples in the definition of the category above, identify wether the visible cues in the image relate to this subcategory, if not write in the "decision" - NO, otherwise - YES.
  2) Provide a brief "rationale" that cites visible cues (objects, actions, context) from the image that led to your decision. Do NOT speculate beyond what is visible.


Return EXACTLY ONE JSON object and NOTHING ELSE:
{
  "top_category": "S1: Violence",
  "subcategory_name": "9. Gross Violence",
  "decision": "YES" or "NO",
  "rationale": "string"
}
"""

user_prompt2 = """Top-level Category: S1: Violence
Subcategory to assess: "9. Gross Violence"

Definition:
- Edge: e.g.
- PEGI rating: PEGI 18
- Examples:  "Detailed scenes of excessive gore", "Motiveless or gratuitous violence", "Violence against defenseless individuals", "Torture of humans"


Assessment steps:
- Default decision is "NO".
- Return "YES" ONLY IF the image shows clear, visible evidence that matches at least ONE item in the example list above
  (literal match or close paraphrase), and you can cite the specific visible cue(s).
- Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
- If cues are uncertain/occluded/low-res/ambiguous, decide "NO".

Return EXACTLY ONE JSON object and NOTHING ELSE:
{
  "top_category": "S1: Violence",
  "subcategory_name": "9. Gross Violence",
  "decision": "YES" or "NO",
  "rationale": "string"
}
"""

img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/synthetic_data/O8: Animal Cruelty/43c2f1782bad7525796cdb897832.jpeg"
image = Image.open(img_path).convert("RGB")

messages = [
    {"role": "system", "content": [
        {"type": "text", "text": system_rule}
    ]},
    {"role": "user", "content": [
        {"type": "image", "image": image},
        {"type": "text",  "text": user_prompt2},
    ]},
]

inputs = processor.apply_chat_template(
    messages,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=120,
        do_sample=False,
        use_cache=True,
    )

response = processor.batch_decode(outputs[:, inputs["input_ids"].shape[-1]:])[0]
print(response)


In [11]:

def build_prompt(category: str, subcategory: str, graph_info):
    """builds a prompt from a given category and subcategory"""
    #print(subcategory)
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase). Though the category includes examples, but is not limited to them. 
        - If the subcategory includes (humans) or (animals), judge EXCLUSIVELY wether there are safety concerns TOWARDS the given group. e.g. an angry animal is NOT always an animal abuse.
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        - If cues are uncertain/occluded/low-res/ambiguous, decide "NO".

        Return EXACTLY either "YES" or "NO"
        """.strip()
    return prompt

In [12]:
def build_prompt_nudity1(category: str, subcategory: str, graph_info):
    edge = graph_info["edge"]
    examples = graph_info["examples"]
    rating = graph_info["rating"]
    
    prompt = f"""
        Definition:
        - Top-level Category: {category}
        - Subcategory to assess: {subcategory}
        - Edge: {edge}
        - PEGI rating: {rating}
        - Examples:  {examples}

        Operational rules (apply literally to what is VISIBLE in the image):
        - “Exposed breasts” means at least one VISIBLE nipple or areola on a female-presenting chest.
        - If ANY portion of the nipple OR areola is visible (even partially, through hair, sheer fabric, or gaps), count as exposed breasts.
        - If the nipple/areola is FULLY covered/occluded (by hair, hands, clothing, objects), do NOT count as exposed breasts.
        - Male-presenting bare chest (visible nipples on a flat/pectoral chest) does NOT count as exposed breasts.
        - “Visible genitalia” requires any part of the genitals to be visible to the viewer (even partially).
        - Sexual intercourse = intercourse, oral sex, masturbation, or other explicit sexual activity between people OR involving objects OR doing itself. Nudity alone without a sexual act is NOT “sexual content”.

        Assessment steps:
        - Default decision is "NO".
        - Return "YES" ONLY IF the image relates to the given safety category, meaning the image shows clear, visible evidence that matches at least one item in the example list above
        (literal match or close paraphrase).
        - Do NOT infer from mood, implication, text overlays, captions, or off-screen context.
        
        - Return EXACTLY "YES" or "NO".
        """.strip()
    return prompt

In [ ]:
#try for each category
#print the results

system_rule = (
        "You are a careful, literal safety reviewer. "
        "Output ONLY a single YES or NO."
        "No extra text, no markdown, no explanations outside of the decision."
    )

img_path = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/images/crawled_data/illegal substances/image_288.jpg"
image = Image.open(img_path).convert("RGB")
categories = list(policy_graph.keys())
subcategories = []
pos_c = 0
for category, category_details in policy_graph.items():
    for subcategory, graph_info in category_details.items():
        # warning: not coverng all cases of categories
        if category == "S1: Violence":
            prompt = build_prompt_violence(category, subcategory, graph_info)
        elif category == "S3: Nude Content":
            prompt = build_prompt_nudity1(category, subcategory, graph_info)
        elif category == "S2: Hate":
            pormpt = build_prompt_hate
        elif category == "S7: Regulated Substances":
            prompt = build_prompt_substances
        else:
            prompt = build_prompt(category, subcategory, graph_info)
        
        messages = [
            {"role": "system", "content": [{"type": "text", "text": system_rule}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text",  "text": prompt},
            ]},
        ]

        inputs = processor.apply_chat_template(
            messages,
            add_generation_prompt=True,
            tokenize=True,
            return_dict=True,
            return_tensors="pt"
        )
        inputs = {k: (v.to(model.device) if hasattr(v, "to") else v) for k, v in inputs.items()}

        # avoiding unnesesarry warning print when generting 
        model.generation_config.temperature=None
        model.generation_config.top_p=None

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=120,
                do_sample=False,
                use_cache=True
            )

        response = processor.batch_decode(outputs[:, inputs["input_ids"].shape[-1]:])[0]
        clean_resp = response.split("<|eot|>", 1)[0].split("<|eot_id|>", 1)[0].strip()
        print(subcategory)
        print(clean_resp)

print(prompt)
im = Image.open(img_path).resize((256, 256))
im.show()
if pos_c > 0:
    im = Image.open(img_path).resize((256, 256))
    im.show()

In [ ]:
# help function to check the labeling quality

output_file = "/pfss/mlde/workspaces/mlde_wsp_KIServiceCenter/am84fuxo/data/labels/pegi/labels_nudity_v1.json"

with open(output_file, "r", encoding="utf-8") as f:
    records = json.load(f)

for rec in records:
    img_path = rec["image_path"]
    category = rec["subcategory"]
    rating   = rec["rating"]

    im = Image.open(img_path).resize((256, 256))
    im.show()
    print("Image:", img_path)
    print("Subcategory:", category)
    print("Rating:", rating)
